# GDF-Netv5 Fold 0 Training (100 Epochs) & Official Blind Test Benchmark (ISIC-2018)

- **Architecture:** GDF-Netv5 (Hierarchical PVTv2-B4 + REGS Edge Stream + Multi-Scale Frequency Filter + Mask-Guided LGAG + Cascaded MSCAM Decoder)
- **Input Resolution:** Native $512 	imes 512$
- **Strategy:** 100 Epochs on Fold 0 (Stratified 80:20 Split) with Cosine Annealing LR & Mixed Precision (AMP fp16)
- **Evaluation:** Official 1,000 Blind Test Images of ISIC-2018 Challenge Task 1 (Single-View & 4-View Flip TTA)


In [ ]:
!pip install -q timm albumentations scipy

import os, sys, glob, json, math, time, cv2, random, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import timm
from scipy.ndimage import distance_transform_edt
from sklearn.model_selection import KFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Active Device: {device} | Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
torch.backends.cudnn.benchmark = True

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)


In [ ]:
DATA_SEARCH_ROOT = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/results_gdfnet_v5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def isic_id(path):
    base = os.path.basename(path).replace("_segmentation", "")
    return os.path.splitext(base)[0]

def find_dir_by_name(target_name, base_search=DATA_SEARCH_ROOT):
    for root, dirs, files in os.walk(base_search):
        if os.path.basename(root).lower() == target_name.lower():
            return root
    return None

def pair_data(img_dir, gt_dir):
    if not img_dir or not gt_dir:
        return [], []
    imgs = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    msks = sorted(glob.glob(os.path.join(gt_dir, "*.png")))
    id_map = {isic_id(m): m for m in msks}
    paired_x, paired_y = [], []
    for im in imgs:
        i_id = isic_id(im)
        if i_id in id_map:
            paired_x.append(im)
            paired_y.append(id_map[i_id])
    return paired_x, paired_y

train_img_dir = find_dir_by_name("ISIC2018_Task1-2_Training_Input")
train_gt_dir  = find_dir_by_name("ISIC2018_Task1_Training_GroundTruth")
test_img_dir  = find_dir_by_name("ISIC2018_Task1-2_Test_Input")
test_gt_dir   = find_dir_by_name("ISIC2018_Task1_Test_GroundTruth")

train_imgs, train_msks = pair_data(train_img_dir, train_gt_dir)
test_imgs, test_msks   = pair_data(test_img_dir, test_gt_dir)

print(f"[+] Train Image Directory: {train_img_dir}")
print(f"[+] Train Mask Directory:  {train_gt_dir}")
print(f"[+] Test Image Directory:  {test_img_dir}")
print(f"[+] Test Mask Directory:   {test_gt_dir}")
print(f"[+] Discovered: {len(train_imgs)} Train images | {len(test_imgs)} Blind Test images")


In [ ]:
class RandomVignettingTransform:
    def __init__(self, p=0.4, min_radius_ratio=0.6, max_radius_ratio=1.0):
        self.p = p
        self.min_radius = min_radius_ratio
        self.max_radius = max_radius_ratio

    def __call__(self, img):
        if random.random() > self.p:
            return img
        h, w = img.shape[:2]
        center_x = w // 2 + random.randint(-int(w * 0.1), int(w * 0.1))
        center_y = h // 2 + random.randint(-int(h * 0.1), int(h * 0.1))
        max_dist = math.hypot(w / 2, h / 2)
        radius = random.uniform(self.min_radius, self.max_radius) * max_dist
        
        y, x = np.ogrid[:h, :w]
        dist_from_center = np.hypot(x - center_x, y - center_y)
        mask = 1.0 - np.clip((dist_from_center - (radius * 0.5)) / (radius * 0.5 + 1e-6), 0.0, 1.0)
        mask = np.dstack([mask] * 3).astype(np.float32)
        return np.clip(img.astype(np.float32) * mask, 0, 255).astype(np.uint8)

class SyntheticHairTransform:
    def __init__(self, p=0.3, max_hairs=6):
        self.p = p
        self.max_hairs = max_hairs

    def __call__(self, img):
        if random.random() > self.p:
            return img
        out = img.copy()
        h, w = out.shape[:2]
        num_hairs = random.randint(1, self.max_hairs)
        for _ in range(num_hairs):
            x1, y1 = random.randint(0, w), random.randint(0, h)
            x2, y2 = random.randint(0, w), random.randint(0, h)
            thickness = random.randint(1, 2)
            color = (random.randint(10, 40), random.randint(10, 40), random.randint(10, 40))
            cv2.line(out, (x1, y1), (x2, y2), color, thickness=thickness)
        return out

IMG_SIZE = 512

class ISICDatasetV5(Dataset):
    def __init__(self, imgs, msks, is_train=True):
        self.imgs = imgs
        self.msks = msks
        self.is_train = is_train
        self.vignetting = RandomVignettingTransform(p=0.4)
        self.hair = SyntheticHairTransform(p=0.3)

        if is_train:
            self.transform = A.Compose([
                A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.6, 1.0), p=0.7),
                A.Resize(height=IMG_SIZE, width=IMG_SIZE),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.6),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
                A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
        else:
            self.transform = A.Compose([
                A.Resize(height=IMG_SIZE, width=IMG_SIZE),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx], cv2.IMREAD_COLOR)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = np.array(Image.open(self.imgs[idx]).convert('RGB'))

        msk = cv2.imread(self.msks[idx], cv2.IMREAD_GRAYSCALE)
        if msk is None:
            msk = np.array(Image.open(self.msks[idx]).convert('L'))
        msk = (msk > 127).astype(np.float32)

        if self.is_train:
            img = self.vignetting(img)
            img = self.hair(img)

        augmented = self.transform(image=img, mask=msk)
        img = np.transpose(augmented['image'].astype(np.float32), (2, 0, 1))
        msk = np.expand_dims(augmented['mask'].astype(np.float32), axis=0)
        return torch.from_numpy(img), torch.from_numpy(msk)


In [ ]:
class HierarchicalBackboneV5(nn.Module):
    def __init__(self, backbone_name="pvt_v2_b4", pretrained=True, in_chans=3):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            features_only=True
        )
        dummy = torch.randn(2, in_chans, 256, 256)
        with torch.no_grad():
            feats = self.backbone(dummy)
        self.channels = [f.shape[1] for f in feats]
        print(f"[+] Hierarchical Backbone: {backbone_name} | Pretrained: {pretrained} | Stage Channels: {self.channels}")

    def forward(self, x):
        return self.backbone(x)

class MultiScaleFrequencyFilterV5(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        c_branch = max(in_channels // 4, 16)
        self.in_proj = nn.Conv2d(in_channels, c_branch * 3, kernel_size=1, bias=False)
        self.norm_in = nn.BatchNorm2d(c_branch * 3)
        self.act = nn.GELU()

        self.dw_1x1 = nn.Conv2d(c_branch, c_branch, kernel_size=1, bias=False)
        self.dw_3x3 = nn.Conv2d(c_branch, c_branch, kernel_size=3, padding=1, groups=c_branch, bias=False)
        self.dw_5x5 = nn.Conv2d(c_branch, c_branch, kernel_size=5, padding=2, groups=c_branch, bias=False)

        self.out_proj = nn.Conv2d(c_branch * 3, in_channels, kernel_size=1, bias=False)
        self.norm_out = nn.BatchNorm2d(in_channels)
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, max(in_channels // 8, 8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(in_channels // 8, 8), in_channels, kernel_size=1),
            nn.Sigmoid()
        )
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(in_channels, 1, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, x):
        residual = x
        proj = self.act(self.norm_in(self.in_proj(x)))
        b1, b2, b3 = torch.chunk(proj, 3, dim=1)
        f1 = self.dw_1x1(b1)
        f2 = self.dw_3x3(b2)
        f3 = self.dw_5x5(b3)
        fused = torch.cat([f1, f2, f3], dim=1)
        fused = self.norm_out(self.out_proj(fused))
        gated = fused * self.channel_gate(fused)
        gated = gated * self.spatial_gate(gated)
        return self.act(residual + gated)

class MaskGuidedLGAG(nn.Module):
    def __init__(self, high_channels, low_channels, out_channels=None, kernel_size=7):
        super().__init__()
        if out_channels is None:
            out_channels = low_channels
        self.high_proj = nn.Sequential(
            nn.Conv2d(high_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.low_proj = nn.Sequential(
            nn.Conv2d(low_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.mask_proj = nn.Sequential(
            nn.Conv2d(1, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        padding = (kernel_size - 1) // 2
        self.grouped_conv = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, padding=padding, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Sigmoid()
        )
        self.out_conv = nn.Sequential(
            nn.Conv2d(low_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, high_feat, low_feat, mask_guidance=None):
        h_high, w_high = high_feat.shape[2:]
        h_low, w_low = low_feat.shape[2:]
        if (h_high, w_high) != (h_low, w_low):
            high_aligned = F.interpolate(high_feat, size=(h_low, w_low), mode='bilinear', align_corners=False)
        else:
            high_aligned = high_feat

        psi = self.high_proj(high_aligned) + self.low_proj(low_feat)
        if mask_guidance is not None:
            if mask_guidance.shape[2:] != (h_low, w_low):
                mask_aligned = F.interpolate(mask_guidance, size=(h_low, w_low), mode='bilinear', align_corners=False)
            else:
                mask_aligned = mask_guidance
            psi = psi + self.mask_proj(mask_aligned)

        attn_weights = self.grouped_conv(psi)
        attended_low = low_feat * attn_weights
        return self.out_conv(attended_low)

class ReverseEdgeGuidanceStream(nn.Module):
    def __init__(self, in_channels_list, edge_channels=64):
        super().__init__()
        self.proj_f1 = nn.Sequential(
            nn.Conv2d(in_channels_list[0], edge_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(edge_channels),
            nn.GELU()
        )
        self.proj_f2 = nn.Sequential(
            nn.Conv2d(in_channels_list[1], edge_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(edge_channels),
            nn.GELU()
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(edge_channels * 2, edge_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(edge_channels),
            nn.GELU(),
            nn.Conv2d(edge_channels, edge_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(edge_channels),
            nn.GELU()
        )
        self.edge_head = nn.Conv2d(edge_channels, 1, kernel_size=1)
        self.edge_gate_head = nn.Sequential(
            nn.Conv2d(edge_channels, 1, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, f1, f2, target_size=(512, 512)):
        f1_proj = self.proj_f1(f1)
        f2_proj = self.proj_f2(f2)
        if f2_proj.shape[2:] != f1_proj.shape[2:]:
            f2_proj = F.interpolate(f2_proj, size=f1_proj.shape[2:], mode='bilinear', align_corners=False)
        cat = torch.cat([f1_proj, f2_proj], dim=1)
        fused = self.fusion(cat)
        edge_raw = self.edge_head(fused)
        edge_pred_full = F.interpolate(edge_raw, size=target_size, mode='bilinear', align_corners=False)
        edge_gate = self.edge_gate_head(fused)
        return edge_pred_full, edge_gate, fused

class MultiScaleConvolutionalAttentionModule(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_c = out_channels // 2
        self.conv1x1 = nn.Conv2d(in_channels, mid_c, kernel_size=1, bias=False)
        self.dw_conv3x3 = nn.Conv2d(mid_c, mid_c, kernel_size=3, padding=1, groups=mid_c, bias=False)
        self.dw_conv5x5 = nn.Conv2d(mid_c, mid_c, kernel_size=5, padding=2, groups=mid_c, bias=False)
        self.out_conv = nn.Sequential(
            nn.Conv2d(mid_c * 3, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.shortcut(x)
        c = self.conv1x1(x)
        c3 = self.dw_conv3x3(c)
        c5 = self.dw_conv5x5(c)
        out = self.out_conv(torch.cat([c, c3, c5], dim=1))
        return out + res

class CascadedMSCAMDecoderBlockV5(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )
        self.mscam = MultiScaleConvolutionalAttentionModule(out_channels, out_channels)
        self.deep_supervision = nn.Conv2d(out_channels, 1, kernel_size=1)

    def forward(self, x, skip):
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        cat = torch.cat([x, skip], dim=1)
        fused = self.fusion_conv(cat)
        feat = self.mscam(fused)
        mask = self.deep_supervision(feat)
        return feat, mask

class GDFNetV5(nn.Module):
    def __init__(self, backbone_name="pvt_v2_b4", pretrained=True, in_chans=3, decoder_channels=64):
        super().__init__()
        self.backbone = HierarchicalBackboneV5(backbone_name=backbone_name, pretrained=pretrained, in_chans=in_chans)
        c1, c2, c3, c4 = self.backbone.channels

        self.freq_filter4 = MultiScaleFrequencyFilterV5(c4)
        self.freq_filter3 = MultiScaleFrequencyFilterV5(c3)
        self.freq_filter2 = MultiScaleFrequencyFilterV5(c2)
        self.freq_filter1 = MultiScaleFrequencyFilterV5(c1)

        self.regs = ReverseEdgeGuidanceStream(in_channels_list=[c1, c2], edge_channels=64)

        self.top_proj = nn.Sequential(
            nn.Conv2d(c4, decoder_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(decoder_channels),
            nn.GELU()
        )
        self.top_mask = nn.Conv2d(decoder_channels, 1, kernel_size=1)

        self.lgag3 = MaskGuidedLGAG(high_channels=decoder_channels, low_channels=c3, out_channels=decoder_channels)
        self.dec3  = CascadedMSCAMDecoderBlockV5(in_channels=decoder_channels, skip_channels=decoder_channels, out_channels=decoder_channels)

        self.lgag2 = MaskGuidedLGAG(high_channels=decoder_channels, low_channels=c2, out_channels=decoder_channels)
        self.dec2  = CascadedMSCAMDecoderBlockV5(in_channels=decoder_channels, skip_channels=decoder_channels, out_channels=decoder_channels)

        self.lgag1 = MaskGuidedLGAG(high_channels=decoder_channels, low_channels=c1, out_channels=decoder_channels)
        self.dec1  = CascadedMSCAMDecoderBlockV5(in_channels=decoder_channels, skip_channels=decoder_channels, out_channels=decoder_channels)

        self.final_edge_fuse = nn.Sequential(
            nn.Conv2d(decoder_channels + 64, decoder_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(decoder_channels),
            nn.GELU()
        )
        self.final_head = nn.Conv2d(decoder_channels, 1, kernel_size=1)

    def forward(self, x):
        h_in, w_in = x.shape[2:]
        f1, f2, f3, f4 = self.backbone(x)

        f4 = self.freq_filter4(f4)
        f3 = self.freq_filter3(f3)
        f2 = self.freq_filter2(f2)
        f1 = self.freq_filter1(f1)

        edge_pred_full, edge_gate, edge_feat = self.regs(f1, f2, target_size=(h_in, w_in))

        d4 = self.top_proj(f4)
        m4 = self.top_mask(d4)

        s3 = self.lgag3(high_feat=d4, low_feat=f3, mask_guidance=m4)
        d3, m3 = self.dec3(d4, s3)

        s2 = self.lgag2(high_feat=d3, low_feat=f2, mask_guidance=m3)
        d2, m2 = self.dec2(d3, s2)

        s1 = self.lgag1(high_feat=d2, low_feat=f1, mask_guidance=m2)
        d1, m1 = self.dec1(d2, s1)

        if edge_feat.shape[2:] != d1.shape[2:]:
            edge_feat_aligned = F.interpolate(edge_feat, size=d1.shape[2:], mode='bilinear', align_corners=False)
        else:
            edge_feat_aligned = edge_feat

        d0 = self.final_edge_fuse(torch.cat([d1, edge_feat_aligned], dim=1))
        out_raw = self.final_head(d0)

        out = F.interpolate(out_raw, size=(h_in, w_in), mode='bilinear', align_corners=False)
        m1 = F.interpolate(m1, size=(h_in, w_in), mode='bilinear', align_corners=False)
        m2 = F.interpolate(m2, size=(h_in, w_in), mode='bilinear', align_corners=False)
        m3 = F.interpolate(m3, size=(h_in, w_in), mode='bilinear', align_corners=False)
        m4 = F.interpolate(m4, size=(h_in, w_in), mode='bilinear', align_corners=False)

        return out, m1, m2, m3, m4, edge_pred_full


In [ ]:
class StructureLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pred, mask):
        weit = 1 + 5 * torch.abs(F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask)
        wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction='none')
        wbce = (weit * wbce).sum(dim=(2, 3)) / (weit.sum(dim=(2, 3)) + 1e-6)

        pred = torch.sigmoid(pred)
        inter = ((pred * mask) * weit).sum(dim=(2, 3))
        union = ((pred + mask) * weit).sum(dim=(2, 3))
        wiou = 1.0 - (inter + 1.0) / (union - inter + 1.0)
        return (wbce + wiou).mean()

def extract_edge_ground_truth(mask):
    dilated = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)
    edge = dilated - eroded
    return torch.clamp(edge, 0.0, 1.0)

class BoundaryEdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, edge_pred, edge_gt):
        pos_weight = (1.0 - edge_gt.mean()).clamp(min=0.8)
        bce = F.binary_cross_entropy_with_logits(edge_pred, edge_gt, pos_weight=pos_weight)
        p = torch.sigmoid(edge_pred)
        inter = (p * edge_gt).sum(dim=(2, 3))
        union = p.sum(dim=(2, 3)) + edge_gt.sum(dim=(2, 3))
        dice_loss = 1.0 - (2.0 * inter + 1.0) / (union + 1.0)
        return bce + dice_loss.mean()

class GDFNetV5CompositeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.structure_loss = StructureLoss()
        self.boundary_loss = BoundaryEdgeLoss()

    def forward(self, outputs, target_mask):
        out, m1, m2, m3, m4, edge_pred = outputs
        edge_gt = extract_edge_ground_truth(target_mask)

        l_out = self.structure_loss(out, target_mask)
        l_m1  = self.structure_loss(m1, target_mask)
        l_m2  = self.structure_loss(m2, target_mask)
        l_m3  = self.structure_loss(m3, target_mask)
        l_m4  = self.structure_loss(m4, target_mask)
        l_edge = self.boundary_loss(edge_pred, edge_gt)

        total_loss = l_out + 0.5 * l_m1 + 0.4 * l_m2 + 0.3 * l_m3 + 0.2 * l_m4 + 0.8 * l_edge
        return total_loss

def compute_surface_distances(gt_binary, pred_binary):
    if not np.any(gt_binary) or not np.any(pred_binary):
        return np.nan, np.nan
    gt_boundary = gt_binary ^ cv2.erode(gt_binary.astype(np.uint8), np.ones((3,3), np.uint8))
    pred_boundary = pred_binary ^ cv2.erode(pred_binary.astype(np.uint8), np.ones((3,3), np.uint8))
    dt_gt = distance_transform_edt(~gt_boundary)
    dt_pred = distance_transform_edt(~pred_boundary)
    dist_pred_to_gt = dt_gt[pred_boundary.astype(bool)]
    dist_gt_to_pred = dt_pred[gt_boundary.astype(bool)]
    all_dists = np.concatenate([dist_pred_to_gt, dist_gt_to_pred])
    hd95 = np.percentile(all_dists, 95)
    asd = np.mean(all_dists)
    return hd95, asd

def evaluate_predictions(gt_binary, pred_binary):
    intersection = np.logical_and(gt_binary, pred_binary).sum()
    total_area = gt_binary.sum() + pred_binary.sum()
    dice = 2.0 * intersection / (total_area + 1e-7) if total_area > 0 else 1.0
    union = np.logical_or(gt_binary, pred_binary).sum()
    iou = intersection / (union + 1e-7) if union > 0 else 1.0
    tp = intersection
    fp = np.logical_and(~gt_binary, pred_binary).sum()
    fn = np.logical_and(gt_binary, ~pred_binary).sum()
    tn = np.logical_and(~gt_binary, ~pred_binary).sum()
    precision = tp / (tp + fp + 1e-7) if (tp + fp) > 0 else 1.0
    sensitivity = tp / (tp + fn + 1e-7) if (tp + fn) > 0 else 1.0
    specificity = tn / (tn + fp + 1e-7) if (tn + fp) > 0 else 1.0
    hd95, asd = compute_surface_distances(gt_binary, pred_binary)
    return {
        'dice': dice, 'iou': iou, 'precision': precision,
        'sensitivity': sensitivity, 'specificity': specificity,
        'hd95': hd95, 'asd': asd
    }


In [ ]:
EPOCHS = 100
BATCH_SIZE = 8
ACCUM_STEPS = 2
LR = 1e-4

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(kf.split(train_imgs))
train_idx, val_idx = fold_splits[0]

fold_train_imgs = [train_imgs[i] for i in train_idx]
fold_train_msks = [train_msks[i] for i in train_idx]
fold_val_imgs   = [train_imgs[i] for i in val_idx]
fold_val_msks   = [train_msks[i] for i in val_idx]

print(f"[*] Fold 0 Setup: {len(fold_train_imgs)} Train samples | {len(fold_val_imgs)} Val samples")

train_loader = DataLoader(ISICDatasetV5(fold_train_imgs, fold_train_msks, is_train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(ISICDatasetV5(fold_val_imgs, fold_val_msks, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = GDFNetV5(backbone_name="pvt_v2_b4", pretrained=True).to(device)
criterion = GDFNetV5CompositeLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()

best_val_dice = 0.0
checkpoint_best = os.path.join(OUTPUT_DIR, "GDFNetV5_fold0_best.pth")
checkpoint_latest = os.path.join(OUTPUT_DIR, "GDFNetV5_fold0_latest.pth")
history_records = []

print(f"\n{'='*60}\n[*] STARTING GDF-NETV5 FOLD 0 TRAINING (100 EPOCHS)\n{'='*60}")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss_scaled = loss / ACCUM_STEPS

        scaler.scale(loss_scaled).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item()

    scheduler.step()
    epoch_loss = running_loss / len(train_loader)
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    # Validation
    model.eval()
    val_dices = []
    total_inter, total_area = 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            with torch.cuda.amp.autocast():
                out, _, _, _, _, _ = model(images)
            probs = torch.sigmoid(out).cpu().numpy()
            targets = masks.numpy()
            preds = (probs > 0.5).astype(np.float32)

            for b in range(preds.shape[0]):
                p_bin = preds[b, 0] > 0.5
                t_bin = targets[b, 0] > 0.5
                inter = np.logical_and(p_bin, t_bin).sum()
                area = p_bin.sum() + t_bin.sum()
                d = 2.0 * inter / (area + 1e-7) if area > 0 else 1.0
                val_dices.append(d)
                total_inter += inter
                total_area += area

    val_macro_dice = np.mean(val_dices)
    val_global_dice = 2.0 * total_inter / (total_area + 1e-7)

    record = {
        'epoch': epoch, 'loss': epoch_loss, 'lr': current_lr,
        'val_macro_dice': val_macro_dice, 'val_global_dice': val_global_dice,
        'time_sec': elapsed
    }
    history_records.append(record)

    log_line = f"Epoch {epoch:03d}/{EPOCHS} [{elapsed:.1f}s] | Loss: {epoch_loss:.4f} | LR: {current_lr:.6f} | Val Macro Dice: {val_macro_dice:.4f} | Val Global Dice: {val_global_dice:.4f}"
    if val_macro_dice > best_val_dice:
        best_val_dice = val_macro_dice
        torch.save(model.state_dict(), checkpoint_best)
        print(f"{log_line} -> [*] NEW BEST ({best_val_dice:.4f})")
    else:
        print(log_line)

    if epoch % 5 == 0 or epoch == EPOCHS:
        torch.save(model.state_dict(), checkpoint_latest)
        pd.DataFrame(history_records).to_csv(os.path.join(OUTPUT_DIR, "training_history_fold0.csv"), index=False)

print(f"\n[+] Training Complete! Best Val Macro Dice: {best_val_dice:.4f}")


In [ ]:
print(f"\n{'='*60}\n[*] STARTING OFFICIAL 1,000 BLIND TEST BENCHMARK\n{'='*60}")

model = GDFNetV5(backbone_name="pvt_v2_b4", pretrained=False).to(device)
model.load_state_dict(torch.load(checkpoint_best, map_location=device))
model.eval()

test_loader = DataLoader(ISICDatasetV5(test_imgs, test_msks, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

def predict_single_view(model, images):
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            out, _, _, _, _, _ = model(images)
        return torch.sigmoid(out)

def predict_tta(model, images):
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            # 1. Normal
            out1, _, _, _, _, _ = model(images)
            p1 = torch.sigmoid(out1)
            # 2. Horizontal Flip
            out2, _, _, _, _, _ = model(torch.flip(images, dims=[3]))
            p2 = torch.flip(torch.sigmoid(out2), dims=[3])
            # 3. Vertical Flip
            out3, _, _, _, _, _ = model(torch.flip(images, dims=[2]))
            p3 = torch.flip(torch.sigmoid(out3), dims=[2])
            # 4. H+V Flip
            out4, _, _, _, _, _ = model(torch.flip(images, dims=[2, 3]))
            p4 = torch.flip(torch.sigmoid(out4), dims=[2, 3])
        return (p1 + p2 + p3 + p4) / 4.0

single_records = []
tta_records = []

for step, (images, masks) in enumerate(tqdm(test_loader, desc="Testing 1,000 Blind Test Images")):
    images = images.to(device)
    
    # 1. Single View
    p_sv = predict_single_view(model, images).cpu().numpy()
    # 2. 4-View TTA
    p_tta = predict_tta(model, images).cpu().numpy()
    
    targets = masks.numpy()
    batch_start = step * BATCH_SIZE

    for b in range(images.shape[0]):
        im_idx = batch_start + b
        im_name = os.path.basename(test_imgs[im_idx])
        t_bin = targets[b, 0] > 0.5

        # SV Eval
        res_sv = evaluate_predictions(t_bin, p_sv[b, 0] > 0.5)
        res_sv['image_id'] = im_name
        single_records.append(res_sv)

        # TTA Eval
        res_tta = evaluate_predictions(t_bin, p_tta[b, 0] > 0.5)
        res_tta['image_id'] = im_name
        tta_records.append(res_tta)

df_sv = pd.DataFrame(single_records)
df_tta = pd.DataFrame(tta_records)

# Calculate Challenge Metric J_thresh (Threshold Jaccard: 0 if IoU < 0.65)
def summarize_results(df, mode_name="SingleView"):
    ious = df['iou'].values
    j_thresh = np.where(ious >= 0.65, ious, 0.0)
    failed_cases = np.sum(ious < 0.65)
    
    summary = {
        'Mode': mode_name,
        'Mean_Dice': df['dice'].mean(),
        'Median_Dice': df['dice'].median(),
        'Min_Dice': df['dice'].min(),
        'Max_Dice': df['dice'].max(),
        'Std_Dice': df['dice'].std(),
        'Mean_mIoU': df['iou'].mean(),
        'Median_mIoU': df['iou'].median(),
        'J_thresh': np.mean(j_thresh),
        'Failed_Cases_J_below_0.65': int(failed_cases),
        'Precision': df['precision'].mean(),
        'Sensitivity': df['sensitivity'].mean(),
        'Specificity': df['specificity'].mean(),
        'HD95_Mean': df['hd95'].dropna().mean(),
        'ASD_Mean': df['asd'].dropna().mean()
    }
    return pd.DataFrame([summary])

summary_sv = summarize_results(df_sv, "SingleView")
summary_tta = summarize_results(df_tta, "4View_TTA")

summary_sv.to_csv(os.path.join(OUTPUT_DIR, "benchmark_summary_ISIC2018_Fold0_SingleView.csv"), index=False)
summary_tta.to_csv(os.path.join(OUTPUT_DIR, "benchmark_summary_ISIC2018_Fold0_TTA.csv"), index=False)
df_tta.to_csv(os.path.join(OUTPUT_DIR, "benchmark_raw_ISIC2018_Fold0_TTA.csv"), index=False)

# Copy to /kaggle/working/ for top-level download
shutil.copy(os.path.join(OUTPUT_DIR, "benchmark_summary_ISIC2018_Fold0_SingleView.csv"), "/kaggle/working/benchmark_summary_ISIC2018_Fold0_SingleView.csv")
shutil.copy(os.path.join(OUTPUT_DIR, "benchmark_summary_ISIC2018_Fold0_TTA.csv"), "/kaggle/working/benchmark_summary_ISIC2018_Fold0_TTA.csv")
shutil.copy(os.path.join(OUTPUT_DIR, "training_history_fold0.csv"), "/kaggle/working/training_history_fold0.csv")
shutil.copy(checkpoint_best, "/kaggle/working/GDFNetV5_fold0_best.pth")

print(f"\n{'='*70}\n[*] ISIC-2018 OFFICIAL 1,000 BLIND TEST RESULTS (FOLD 0)\n{'='*70}")
print("\n--- Single-View Benchmark ---")
print(summary_sv.to_string(index=False))
print("\n--- 4-View Flip TTA Benchmark ---")
print(summary_tta.to_string(index=False))
print(f"\n[+] All outputs saved to /kaggle/working/ successfully!")
